In [1]:
import os
import sys
import argparse

sys.path.append(os.path.expanduser("~/websites/mapedia"))

import numpy as np
import pandas as pd
import torch
import geopandas as gpd
from torch.utils.tensorboard import SummaryWriter

from src.processing import (
    aggregate_speed_matrix, nlanes_to_class_np, oneway_to_class_np,
    highway_to_class_np, ZScaler, build_line_graph_edge_index,
    build_split_data, degree_stats
)
from src.models.multi_attr_gat import MultiAttrGAT
from src.models.plotting import plot_avgspeed_nans_per_bin, plot_results
from src.models.losses import (
    get_optimizer, corrupt_inputs_with_flags, compute_losses,
    evaluate_losses_only, compute_metrics, evaluate_with_masks
)
from src.models.masking import make_fixed_masks, bernoulli_mask
from src.models.saveing import save_checkpoint

In [2]:
from dataclasses import dataclass
from typing import Optional, Literal

@dataclass
class TrainConfig:
    city:       str            = "jakarta"                      # City name (must match parquet/npy filenames). Required — set before use.
    device:     Optional[str]  = None                    # 'cpu', 'cuda', 'cuda:0', etc. Defaults to cuda if available.
    data_dir:   str            = "./data/raw_data/"      # Directory containing parquet and npy files
    pyg_data_dir: str          = "./data/pyg_data/"
    plots_dir:  str            = "./plots/single-city/"  # Directory to save plots

    epochs:     int            = 500                     # Number of training epochs
    p_mask:     float          = 0.30                    # Masking probability
    eval_every: int            = 1                       # Evaluate metrics every N epochs
    seed:       int            = 42                      # Random seed

    train_frac: float          = 0.85                    # Fraction of data for training
    val_frac:   float          = 0.05                    # Fraction of data for validation
    test_frac:  float          = 0.10                    # Fraction of data for test
    split_axis: Literal["lat", "lon"] = "lat"            # Spatial split axis

    # resume:     Optional[str]  = N
        
args = TrainConfig()

In [3]:


# Device setup — must happen before any torch operations
if args.device is not None:
    device = torch.device(args.device)
    # Set CUDA_VISIBLE_DEVICES only when a specific cuda index is given (e.g. cuda:2)
    if args.device.startswith("cuda:"):
        # gpu_index = args.device.split(":")[1]
        # os.environ["CUDA_VISIBLE_DEVICES"] = gpu_index
        device = torch.device(args.device)  # after restricting visibility, always "cuda"
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"City   : {args.city}")
print(f"Device : {device}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version   : {torch.version.cuda}")

# Reproducibility
np.random.seed(args.seed)
torch.manual_seed(args.seed)

os.makedirs(args.plots_dir, exist_ok=True)

with np.load(f'./data/pyg_data/{args.city}_masked_data.npz') as data:
    # Safely loads arrays in the exact order they were saved
    avg_speed_flat, width_raw, min_raw, max_raw, road_type_raw, nlanes_raw, oneway_raw, length_raw = [
        data[f] for f in data.files
    ]
with np.load(f'./data/pyg_data/{args.city}_masks.npz') as data:
    # Safely loads arrays in the exact order they were saved
    avg_speed_mask, width_mask, min_mask, max_mask, road_type_mask, nlanes_mask, oneway_mask = [
        data[f] for f in data.files
    ]
with np.load(f'./data/pyg_data/{args.city}_true_data.npz') as data:
    # Safely loads arrays in the exact order they were saved
    avg_speed_flat_true, width_true, min_true, max_true, road_type_true, nlanes_true, oneway_true = [
        data[f] for f in data.files
    ]

train_idx = np.load(os.path.join(args.pyg_data_dir, f"{args.city}_train_idx.npy"))
val_idx = np.load(os.path.join(args.pyg_data_dir, f"{args.city}_val_idx.npy"))
test_idx = np.load(os.path.join(args.pyg_data_dir, f"{args.city}_test_idx.npy"))
                    
edges = gpd.read_parquet(os.path.join(args.pyg_data_dir, f"{args.city}_edges.parquet"))
N         = len(edges)

avg_speed_flat, width_raw, min_raw, max_raw, road_type_raw, nlanes_raw, oneway_raw = avg_speed_flat_true, width_true, min_true, max_true, road_type_true, nlanes_true, oneway_true


City   : jakarta
Device : cuda
CUDA available : True
PyTorch version: 2.5.1
CUDA version   : 11.8


In [4]:
# Road Attributes: Road Type, One way, Max Speed, Min Speed, Width, N lanes
X = np.concatenate([
    length_raw.reshape((-1, 1)), 
    width_raw.reshape((-1, 1)), 
    max_raw.reshape((-1, 1)), 
    min_raw.reshape((-1, 1)), 
    road_type_raw.reshape((-1, 1)),
    nlanes_raw.reshape((-1, 1)),
    oneway_raw.reshape((-1, 1)),
    avg_speed_flat], axis=1)
X.shape

(512991, 19)

In [5]:
dummy_mask = np.zeros(width_mask.shape, dtype=bool)

In [6]:
test_mask = np.concatenate([np.column_stack([dummy_mask, width_mask, max_mask, min_mask, road_type_mask, nlanes_mask, oneway_mask]), avg_speed_mask], axis=1)

In [7]:
X_train = X[train_idx]
X_val = X[val_idx]
X_test = X[test_idx]

In [8]:
X_train.shape

(436042, 19)

In [9]:
k = 5_000
idx_max = np.argpartition((~np.isnan(X_train)).sum(axis=1), -k)[-k:]
X_train_knn = X_train[idx_max]

In [10]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
X_train_knn_imputed = imputer.fit_transform(X_train_knn)

In [11]:
X_test_masked = X_test.copy()
X_test_masked[test_mask] = np.nan

In [12]:
X_test_imputed = imputer.transform(X_test_masked)

In [13]:
pred_X = X_test_imputed
pred_X[:, 2:4] = np.round(pred_X[:, 2:4] / 5) * 5
pred_X[:, 4:7] = np.round(pred_X[:, 4:7])

In [14]:
true_X = X_test

In [15]:
from sklearn.metrics import mean_absolute_error
sub_test_mask = test_mask

In [16]:
# Maximum Speed Error (MAE)
A = 2
attr_true_X = true_X[:, A][sub_test_mask[:, A]]
attr_pred_X = pred_X[:, A][sub_test_mask[:, A]]
mean_absolute_error(attr_pred_X, attr_true_X)

18.467432

In [17]:
np.stack([attr_pred_X, attr_true_X]).T

array([[20., 20.],
       [45., 20.],
       [50., 30.],
       [40., 20.],
       [45., 30.],
       [25., 30.],
       [50., 20.],
       [35., 30.],
       [40., 20.],
       [25., 30.],
       [30., 30.],
       [25., 30.],
       [25., 30.],
       [30.,  5.],
       [25.,  5.],
       [25., 30.],
       [40.,  5.],
       [30.,  5.],
       [30.,  5.],
       [30., 30.],
       [30., 30.],
       [65., 10.],
       [55., 80.],
       [25., 10.],
       [85., 80.],
       [25.,  5.],
       [25.,  5.],
       [25., 10.],
       [65., 80.],
       [30.,  5.],
       [55., 10.],
       [35.,  5.],
       [35., 10.],
       [30., 20.],
       [30., 20.],
       [25., 30.],
       [70., 30.],
       [25., 30.],
       [45., 30.],
       [75., 20.],
       [35., 60.],
       [65., 60.],
       [40., 60.],
       [50., 60.],
       [35., 20.],
       [55., 60.],
       [60., 60.],
       [35., 60.],
       [40., 60.],
       [30., 20.],
       [35., 60.],
       [45., 60.],
       [25.,

In [18]:
# Minimum Speed Error (MAE)
A = 3
attr_true_X = true_X[:, A][sub_test_mask[:, A]]
attr_pred_X = pred_X[:, A][sub_test_mask[:, A]]
mean_absolute_error(attr_pred_X, attr_true_X)

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [19]:
# Width Error (MAE)
A = 1
attr_true_X = true_X[:, A][sub_test_mask[:, A]]
attr_pred_X = pred_X[:, A][sub_test_mask[:, A]]
mean_absolute_error(attr_pred_X, attr_true_X)

1.5356627

In [21]:
# Road Type Error (accuracy)
from sklearn.metrics import accuracy_score, f1_score
A = 4
attr_true_X = true_X[:, A][sub_test_mask[:, A]]
attr_pred_X = pred_X[:, A][sub_test_mask[:, A]]
accuracy_score(attr_pred_X, attr_true_X), f1_score(attr_pred_X, attr_true_X, average='macro')

(0.4106099402534305, 0.2239123580352981)

In [33]:
# Lanes Error (MAE)
A = 5
attr_true_X = true_X[:, A][sub_test_mask[:, A]]
attr_pred_X = pred_X[:, A][sub_test_mask[:, A]]
attr_pred_X[attr_pred_X > 3] = 3
attr_true_X[attr_true_X > 3] = 3
accuracy_score(attr_pred_X, attr_true_X), mean_absolute_error(attr_pred_X, attr_true_X), f1_score(attr_pred_X, attr_true_X, average='macro')

(0.6311886586695747, 0.3992003, 0.4621653664711869)

In [ ]:
np.stack([attr_pred_X, attr_true_X]).T

array([[2., 2.],
       [1., 2.],
       [2., 2.],
       ...,
       [1., 1.],
       [1., 1.],
       [1., 1.]], dtype=float32)

In [30]:
np.unique(attr_pred_X)

array([1., 2., 3., 4.], dtype=float32)

In [31]:
np.unique(attr_true_X)

array([1., 2., 3., 4., 5., 6.], dtype=float32)

In [ ]:
# oneway Error (MAE)
from sklearn.metrics import accuracy_score, roc_auc_score
# attr_test_mask = np.zeros(sub_test_mask.shape, dtype=bool)
# attr_test_mask[:, 4] = sub_test_mask[:, 4]
A = 6
attr_true_X = true_X[:, A][sub_test_mask[:, A]]
attr_pred_X = pred_X[:, A][sub_test_mask[:, A]]
accuracy_score(attr_pred_X, attr_true_X), roc_auc_score(attr_pred_X, attr_true_X)

(0.6298710601719197, 0.6017510807384473)

In [23]:
# avg_speed Error (MAE)
A = 7
attr_true_X = true_X[:, A:][sub_test_mask[:, A:]]
attr_pred_X = pred_X[:, A:][sub_test_mask[:, A:]]
mean_absolute_error(attr_pred_X, attr_true_X)

1.7446027

In [58]:
np.stack([attr_pred_X, attr_true_X]).T

array([[ 2.7575183,  3.7712421],
       [ 4.953551 ,  3.809697 ],
       [10.308336 , 11.73927  ],
       ...,
       [ 4.6395617,  6.155    ],
       [ 5.162856 ,  7.419206 ],
       [ 4.7013984,  3.71     ]], dtype=float32)

In [13]:
(~np.isnan(speed_matrix)).sum()

876104

In [14]:
order

array([215327, 163733, 397276, ..., 482587, 483353, 482588])

In [ ]:



avg_scaler = ZScaler()
len_scaler = ZScaler()
wid_scaler = ZScaler()
max_scaler = ZScaler()
min_scaler = ZScaler()

avg_scaler.fit(avg_speed_flat[train_idx])
len_scaler.fit(length_log[train_idx])
wid_scaler.fit(width_raw[train_idx])
max_scaler.fit(max_raw[train_idx])
min_scaler.fit(min_raw[train_idx])

avg_speed_z = avg_scaler.transform(avg_speed_flat).astype(np.float32)
length_z    = len_scaler.transform(length_log).astype(np.float32)
width_z     = wid_scaler.transform(width_raw).astype(np.float32)
max_z       = max_scaler.transform(max_raw).astype(np.float32)
min_z       = min_scaler.transform(min_raw).astype(np.float32)

avg_speed_missing = np.isnan(avg_speed_flat).astype(np.float32)
width_missing     = np.isnan(width_raw).astype(np.float32)
length_missing    = np.zeros(N, dtype=np.float32)
max_missing       = np.isnan(max_raw).astype(np.float32)
min_missing       = np.isnan(min_raw).astype(np.float32)

avg_speed_z = np.nan_to_num(avg_speed_z, nan=0.0)
width_z     = np.nan_to_num(width_z,     nan=0.0)
max_z       = np.nan_to_num(max_z,       nan=0.0)
min_z       = np.nan_to_num(min_z,       nan=0.0)

x_cont_all = np.column_stack([
    length_z,           # (N,)
    width_z,            # (N,)
    max_z,              # (N,)
    min_z,              # (N,)
    avg_speed_z,        # (N, 12)
    length_missing,     # (N,)
    width_missing,      # (N,)
    max_missing,        # (N,)
    min_missing,        # (N,)
    avg_speed_missing,  # (N, 12)
    # mask flags mirror missing flags so truly-missing inputs look the same
    # in training and at inference (see build_x_cont in 3_predict_on_graphs)
    np.zeros(N, dtype=np.float32),        # len_mask
    width_missing,                        # wid_mask
    max_missing,                          # max_mask
    min_missing,                          # min_mask
    avg_speed_missing,                    # avg_speed_mask
]).astype(np.float32)

# =========================
# 7) Build graph splits
# =========================
LANES_MASK_ID  = 4 # nlanes classes: 1,2,3 = lane count, 0 = more than 3 lanes + MASK=4 + MISSING=5
LANES_MISS_ID  = 5
ONEWAY_MASK_ID = 2 # 0,1 + MASK=2 + MISSING=3
ONEWAY_MISS_ID = 3

edges = edges.reset_index().rename(columns={"index": "idx"})

edge_index_full = build_line_graph_edge_index(
    edges, u_col="source", v_col="target", eid_col="idx"
)

ei   = edge_index_full
pairs = (
    ei[0].cpu().numpy().astype(np.int64) * (ei.max().item() + 1)
    + ei[1].cpu().numpy().astype(np.int64)
)
dup = len(pairs) - len(np.unique(pairs))
print(f"Duplicate directed edges in line-graph edge_index: {dup}")

edge_index_full_np = edge_index_full.cpu().numpy()

# Regression targets are z-scored (NaN preserved). Training on raw km/h with
# SmoothL1 (beta=1) puts almost every error in the linear regime → constant
# gradients and median-collapse; z-scored targets keep the loss quadratic over
# most of the range. Predictions are decoded back with the saved scalers.
y_avg_speed_all = avg_scaler.transform(avg_speed_flat).astype(np.float32)
y_highway_all   = edges["highway_id"].to_numpy(dtype=np.int64)
y_nlanes_all    = edges["nlanes_cls"].to_numpy(dtype=np.int64)
y_oneway_all    = edges["oneway"].to_numpy(dtype=np.float32)
y_width_all     = wid_scaler.transform(width_raw).astype(np.float32)
y_max_all       = max_scaler.transform(max_raw).astype(np.float32)
y_min_all       = min_scaler.transform(min_raw).astype(np.float32)

nlanes_in_all = edges["nlanes_cls"].to_numpy(dtype=np.int64)
nlanes_in_all = np.where(nlanes_in_all == -1, LANES_MISS_ID, nlanes_in_all).astype(np.int64)

oneway_in_all = edges["oneway"].to_numpy(dtype=np.float32)
oneway_in_all = np.where(np.isnan(oneway_in_all), ONEWAY_MISS_ID, oneway_in_all).astype(np.int64)

split_kwargs = dict(
    N=N,
    edge_index_full_np=edge_index_full_np,
    x_cont_all=x_cont_all,
    y_highway_all=y_highway_all,
    nlanes_in_all=nlanes_in_all,
    oneway_in_all=oneway_in_all,
    y_nlanes_all=y_nlanes_all,
    y_oneway_all=y_oneway_all,
    y_width_all=y_width_all,
    y_max_all=y_max_all,
    y_min_all=y_min_all,
    y_avg_speed_all=y_avg_speed_all,
    device=device,
)

data_train = build_split_data(train_idx, **split_kwargs)
data_val   = build_split_data(val_idx,   **split_kwargs)
data_test  = build_split_data(test_idx,  **split_kwargs)

print(f"Train graph: {data_train.num_nodes} nodes | {data_train.edge_index.shape[1]} edges")
print(f"Val graph:   {data_val.num_nodes} nodes | {data_val.edge_index.shape[1]} edges")
print(f"Test graph:  {data_test.num_nodes} nodes | {data_test.edge_index.shape[1]} edges")
print("Degree stats:")
print("  Train:", degree_stats(data_train))
print("  Val:  ", degree_stats(data_val))
print("  Test: ", degree_stats(data_test))

print('Availability of avg_speed:')
print('train non nans: ', np.sum(~np.isnan(data_train.y_avg_speed.cpu().numpy())))
print('val non nans: ', np.sum(~np.isnan(data_val.y_avg_speed.cpu().numpy())))
print('test non nans: ', np.sum(~np.isnan(data_test.y_avg_speed.cpu().numpy())))

print('Availability of min_speed:')
print('train non nans: ', np.sum(~np.isnan(data_train.y_min.cpu().numpy())))
print('val non nans: ', np.sum(~np.isnan(data_val.y_min.cpu().numpy())))
print('test non nans: ', np.sum(~np.isnan(data_test.y_min.cpu().numpy())))

# =========================
# 8) Model
# =========================
num_highway = len(hwy2id)
HIGHWAY_UNK_ID = hwy2id[UNK_TOKEN]
# MAE multipliers: metrics computed in z-space × sd = original units (m, km/h)
mae_scale = {"wid": wid_scaler.sd, "max": max_scaler.sd, "min": min_scaler.sd, "avg": avg_scaler.sd}
model = MultiAttrGAT(num_highway=num_highway, cont_dim=48).to(device)
optimizer = get_optimizer(model.parameters())

start_epoch = 1
if args.resume:
    ckpt = torch.load(args.resume, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model_state"])
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt.get("epoch", 0) + 1
    print(f"Resumed from {args.resume}  (epoch {start_epoch - 1} → continuing from {start_epoch})")

_script_dir = os.path.dirname(os.path.abspath(__file__))
tb_log_dir = os.path.join(_script_dir, "tb_logs", args.city)
writer = SummaryWriter(log_dir=tb_log_dir)
print(f"TensorBoard logs → {tb_log_dir}")

# TODO: Remove the p_mask here, keep all validation as possible
val_masks_fixed = make_fixed_masks(data_val, p_mask=args.p_mask, seed=999, hwy_unk_id=HIGHWAY_UNK_ID)

history = {
    "epoch":        [],
    "train_total":  [],
    "val_total":    [],
    "train_losses": {k: [] for k in ["hwy", "lan", "onw", "wid", "max", "min", "avg"]},
    "val_losses":   {k: [] for k in ["hwy", "lan", "onw", "wid", "max", "min", "avg"]},
    "metric_epoch": [],
    "train_metrics": {k: [] for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]},
    "val_metrics":   {k: [] for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]},
    "log_vars":     [],
}

# # Column index constants
# CONT_LENGTH_COL   = 0
# CONT_WIDTH_COL    = 1
# CONT_MAX_COL      = 2
# CONT_MIN_COL      = 3
# CONT_AVG_START    = 4   # avg_speed_z occupies columns 4-15 (12 slots)
# CONT_AVG_END      = 16  # exclusive


# CONT_LENMISS_COL  = 4
# CONT_WIDMISS_COL  = 5
# CONT_MAXMISS_COL  = 6
# CONT_MINMISS_COL  = 7
# CONT_AVGMISS_START = 20  # avg_speed_missing: columns 20-31
# CONT_AVGMISS_END   = 32

# CONT_LENMASK_COL  = 8
# CONT_WIDMASK_COL  = 9
# CONT_MAXMASK_COL  = 10
# CONT_MINMASK_COL  = 11
# CONT_AVGMASK_START = 36  # avg_speed_mask: columns 36-47
# CONT_AVGMASK_END   = 48

# =========================
# 9) Training loop
# =========================
plot_avgspeed_nans_per_bin(data_train, os.path.join(args.plots_dir, "avgspeed_nans_per_bin.png"))

for epoch in range(start_epoch, args.epochs + 1):
    model.train()
    optimizer.zero_grad()

    n = data_train.num_nodes

    valid_hwy = (data_train.y_highway != HIGHWAY_UNK_ID)  # never supervise UNK
    valid_lan = (data_train.y_nlanes != -1)
    valid_onw = ~torch.isnan(data_train.y_oneway)
    valid_wid = ~torch.isnan(data_train.y_width)
    valid_max = ~torch.isnan(data_train.y_max)
    valid_min = ~torch.isnan(data_train.y_min)
    valid_avg = ~torch.isnan(data_train.y_avg_speed)

    train_masks = {
        "hwy": bernoulli_mask(valid_hwy, args.p_mask),
        "lan": bernoulli_mask(valid_lan, args.p_mask),
        "onw": bernoulli_mask(valid_onw, args.p_mask),
        "wid": bernoulli_mask(valid_wid, args.p_mask),
        "max": bernoulli_mask(valid_max, args.p_mask),
        "min": bernoulli_mask(valid_min, args.p_mask),
        "avg": bernoulli_mask(valid_avg, args.p_mask),
    }

    x_cont, highway_in, nlanes_in, oneway_in = corrupt_inputs_with_flags(
        data_train, train_masks, HIGHWAY_UNK_ID
    )

    pred = model(x_cont, highway_in, nlanes_in, oneway_in, data_train.edge_index)

    total_loss, losses = compute_losses(pred, data_train, train_masks, model, device)
    total_loss.backward()
    optimizer.step()

    val_total, val_losses = evaluate_losses_only(
        model, data_val, val_masks_fixed, device, HIGHWAY_UNK_ID
    )

    history["epoch"].append(epoch)
    history["train_total"].append(total_loss.item())
    history["val_total"].append(val_total)
    for k in ["hwy", "lan", "onw", "wid", "max", "min", "avg"]:
        history["train_losses"][k].append(losses[k].item())
        history["val_losses"][k].append(val_losses[k])
    history["log_vars"].append(model.log_vars.detach().cpu().numpy().copy())

    # ── TensorBoard: losses (every epoch) ────────────────────────────────
    writer.add_scalar("loss/train_total", total_loss.item(), epoch)
    writer.add_scalar("loss/val_total",   val_total,         epoch)
    for k in ["hwy", "lan", "onw", "wid", "max", "min", "avg"]:
        writer.add_scalar(f"loss_train/{k}", losses[k].item(), epoch)
        writer.add_scalar(f"loss_val/{k}",   val_losses[k],    epoch)
    log_vars_now = model.log_vars.detach().cpu().numpy()
    for i, (k, lv) in enumerate(zip(["hwy", "lan", "onw", "wid", "max", "min", "avg"], log_vars_now)):
        writer.add_scalar(f"log_vars/{k}", float(lv), epoch)

    do_metrics = (epoch == 1) or (epoch % args.eval_every == 0)
    if do_metrics:
        train_metrics = compute_metrics(pred, data_train, train_masks, num_highway, mae_scale)
        _, _, val_metrics = evaluate_with_masks(
            model, data_val, val_masks_fixed, num_highway, device, HIGHWAY_UNK_ID, mae_scale
        )

        history["metric_epoch"].append(epoch)
        for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]:
            history["train_metrics"][k].append(train_metrics[k])
            history["val_metrics"][k].append(val_metrics[k])

        # ── TensorBoard: metrics (every eval_every epochs) ────────────────
        for metric_key in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]:
            writer.add_scalar(f"train/{metric_key}", train_metrics[metric_key], epoch)
            writer.add_scalar(f"val/{metric_key}",   val_metrics[metric_key],   epoch)

        log_vars = model.log_vars.detach().cpu().numpy()
        print(
            f"\n{'='*60}\n"
            f"  Epoch {epoch:04d}/{args.epochs}\n"
            f"{'='*60}\n"
            f"  LOSS   total={total_loss.item():.4f}\n"
            f"         hwy={losses['hwy'].item():.3f}  lan={losses['lan'].item():.3f}  onw={losses['onw'].item():.3f}\n"
            f"         wid={losses['wid'].item():.3f}  max={losses['max'].item():.3f}  min={losses['min'].item():.3f}  avg={losses['avg'].item():.3f}\n"
            f"  VAL    hwy_F1={val_metrics['hwy_macro_f1']:.3f}  lan_F1={val_metrics['lan_macro_f1']:.3f}  onw_AUROC={val_metrics['onw_auroc']:.3f}\n"
            f"         wid_MAE={val_metrics['wid_mae_m']:.3f}  max_MAE={val_metrics['max_mae']:.3f}  min_MAE={val_metrics['min_mae']:.3f}  avg_MAE={val_metrics['avg_mae']:.3f}\n"
            f"  VARS   {np.array2string(log_vars, precision=3, separator=', ')}\n"
            f"{'='*60}"
        )

        if epoch > 50:
            masked_max_idx = torch.where(train_masks["max"])[0]
            if len(masked_max_idx) >= 5:
                picks = masked_max_idx[
                    torch.linspace(0, len(masked_max_idx) - 1, 5).long()
                ]
                pred_max = max_scaler.inverse_transform(pred["max_speed"].detach()[picks].cpu().numpy())
                true_max = max_scaler.inverse_transform(data_train.y_max[picks].cpu().numpy())
                print(f"  max_speed sample (masked observed roads, km/h):")
                print(f"  {'road_idx':>10}  {'max_speed':>10}  {'true':>8}  {'|err|':>8}")
                for i, p, t in zip(picks.cpu().numpy(), pred_max, true_max):
                    print(f"  {i:>10}  {p:>10.1f}  {t:>8.1f}  {abs(p - t):>8.1f}")

            truly_missing_idx = torch.where(torch.isnan(data_train.y_max))[0]
            if len(truly_missing_idx) >= 10:
                picks_m = truly_missing_idx[
                    torch.randperm(len(truly_missing_idx), generator=torch.Generator().manual_seed(42))[:10]
                ].sort().values
                pred_missing = max_scaler.inverse_transform(pred["max_speed"].detach()[picks_m].cpu().numpy())
                print(f"  max_speed sample (truly missing roads, km/h):")
                print(f"  {'road_idx':>10}  {'max_speed':>10}")
                for i, p in zip(picks_m.cpu().numpy(), pred_missing):
                    print(f"  {i:>10}  {p:>10.1f}")

# =========================
# 10) Final evaluation
# =========================
train_metrics     = compute_metrics(pred, data_train, train_masks, num_highway, mae_scale)
test_masks_fixed  = make_fixed_masks(data_test, p_mask=args.p_mask, seed=999, hwy_unk_id=HIGHWAY_UNK_ID)
_, _, test_metrics = evaluate_with_masks(model, data_test, test_masks_fixed, num_highway, device, HIGHWAY_UNK_ID, mae_scale)
_, _, val_metrics  = evaluate_with_masks(model, data_val,  val_masks_fixed,  num_highway, device, HIGHWAY_UNK_ID, mae_scale)

print(
    f"\nFinal Results — {args.city}\n"
    f"VAL  : hwy_F1={val_metrics['hwy_macro_f1']:.3f}, lan_F1={val_metrics['lan_macro_f1']:.3f}, "
    f"onw_AUROC={val_metrics['onw_auroc']:.3f}, wid_MAE={val_metrics['wid_mae_m']:.3f}, "
    f"max_MAE={val_metrics['max_mae']:.3f}, min_MAE={val_metrics['min_mae']:.3f}\n"
    f"TEST : hwy_F1={test_metrics['hwy_macro_f1']:.3f}, lan_F1={test_metrics['lan_macro_f1']:.3f}, "
    f"onw_AUROC={test_metrics['onw_auroc']:.3f}, wid_MAE={test_metrics['wid_mae_m']:.3f}, "
    f"max_MAE={test_metrics['max_mae']:.3f}, min_MAE={test_metrics['min_mae']:.3f}"
)

test_masks_fixed = make_fixed_masks(data_test, p_mask=args.p_mask, seed=2025, hwy_unk_id=HIGHWAY_UNK_ID)
test_total, test_losses, test_metrics = evaluate_with_masks(
    model, data_test, test_masks_fixed, num_highway, device, HIGHWAY_UNK_ID, mae_scale
)
print("TEST fixed-mask metrics:", test_metrics)
print("TEST fixed-mask losses:",  test_losses)

print(f"GPU memory — allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# ── TensorBoard: final test metrics ──────────────────────────────────────
for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae", "avg_mae"]:
    writer.add_scalar(f"test/{k}", test_metrics[k], args.epochs)
writer.close()

plot_results(history, os.path.join(args.plots_dir, args.city))

# =========================
# 11) Save checkpoint
# =========================
cont_dim = int(data_train.x_cont.shape[1])
save_checkpoint(
    model=model,
    num_highway=num_highway,
    hwy2id=hwy2id,
    id2hwy=id2hwy,
    HIGHWAY_MASK_ID=HIGHWAY_MASK_ID,
    LANES_MASK_ID=LANES_MASK_ID,
    LANES_MISS_ID=LANES_MISS_ID,
    ONEWAY_MASK_ID=ONEWAY_MASK_ID,
    ONEWAY_MISS_ID=ONEWAY_MISS_ID,
    len_scaler=len_scaler,
    wid_scaler=wid_scaler,
    max_scaler=max_scaler,
    min_scaler=min_scaler,
    avg_scaler=avg_scaler,
    SEED=args.seed,
    P_MASK=args.p_mask,
    city=args.city,
    cont_dim=cont_dim,
    optimizer=optimizer,
    epoch=args.epochs,
)